# Assignment 20: Content Based Recommendation System

Dataset: Open Data 500 US Companies (us_companies.csv)
Kaggle: https://www.kaggle.com/datasets/thegovlab/open-data-500-companies

recommend similar companies using TF-IDF + cosine similarity on company descriptions, then make a streamlit app and deploy on render.

In [1]:
import os
import re
import warnings
import numpy as np
import pandas as pd
import nltk
from nltk.corpus import stopwords
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

warnings.filterwarnings('ignore')
nltk.download('stopwords', quiet=True)
print("Libraries loaded successfully")

Libraries loaded successfully


## PART 1 - Data Preprocessing
### Task 1: Load and Understand Dataset

In [2]:
# multi-path check for dataset across working directories
possible_paths = [
    'data/us_companies.csv', 
    'content-based-recommender/data/us_companies.csv', 
    '../data/us_companies.csv', 
    'us_companies.csv'
]

dataset_path = None
for path in possible_paths:
    if os.path.exists(path):
        dataset_path = path
        break

if dataset_path is None:
    raise FileNotFoundError("Could not locate us_companies.csv in candidate paths.")

df = pd.read_csv(dataset_path)
print(f"Loaded from: {dataset_path} | Shape: {df.shape}")
print(f"Columns: {df.columns.tolist()}")
df.head()

Loaded from: data/us_companies.csv | Shape: (529, 22)
Columns: ['company_name_id', 'company_name', 'url', 'year_founded', 'city', 'state', 'country', 'zip_code', 'full_time_employees', 'company_type', 'company_category', 'revenue_source', 'business_model', 'social_impact', 'description', 'description_short', 'source_count', 'data_types', 'example_uses', 'data_impacts', 'financial_info', 'last_updated']


,company_name_id,company_name,url,year_founded,city,state,country,zip_code,full_time_employees,company_type,...,business_model,social_impact,description,description_short,source_count,data_types,example_uses,data_impacts,financial_info,last_updated
0,3-round-stones-inc,"3 Round Stones, Inc.",http://3RoundStones.com,2010.0,Washington,DC,us,20004.0,1-10,Private,...,"Business to Business, Business to Consumer",NaN,3 Round Stones produces a platform for publish...,Our Open Source platform is used by the Fortun...,NaN,NaN,NaN,[],"3 Round Stones is a profitable, self-funded, w...",2014-11-12 14:44:25.969871
1,48-factoring-inc,48 Factoring Inc.,https://www.48factoring.com,2014.0,Philadelphia,PA,us,19087.0,51-200,Private,...,Business to Business,Small Business Owners,The company mission is to provide finance to s...,48 Factoring Inc. is one of the best financial...,11-50,Business,NaN,"[u'Cost efficiency', u'Job growth', u'Revenue ...",NaN,2015-05-18 11:36:39.902880
2,5psolutions,5PSolutions,www.5psolutions.com,2007.0,Fairfax,VA,us,22003.0,1-10,Private,...,"Business to Business, Business to Consumer, Bu...",NaN,"At 5PSolutions, we wish to make all basic info...",5PSolutions are artisans of mobile platforms.,NaN,NaN,NaN,[],NaN,2014-11-12 15:09:35.485949
3,abt-associates,Abt Associates,abtassoc.com,1965.0,Cambridge,MA,us,2138.0,"1,001-5,000",Private,...,NaN,NaN,"Abt Associates is a mission-driven, internatio...","Abt Associates is a mission-driven, global lea...",101+,NaN,NaN,[],Employee-owned company. $552M/year.,2014-09-11 17:23:21.390289
4,accela,Accela,http://www.accela.com,1999.0,San Ramon,CA,us,94583.0,"501-1,000",Private,...,Business to Consumer,"Citizen engagement and participation, Good gov...",Accela powers thousands of services and millio...,Accela improves citizen engagement by making i...,101+,"Health/Healthcare, Housing, Public Safety",NaN,[],NaN,2014-09-16 16:30:49.172802


In [3]:
# Check total rows, unique company names, duplicates, missing values, and top categories.
key_cols = ['company_name', 'company_category', 'description', 'description_short']
print(f"Total companies: {len(df)} | Unique names: {df['company_name'].nunique()}")
print(f"Duplicate rows: {df.duplicated().sum()}\n")
print("Missing values:\n", df[key_cols].isnull().sum())
print("\nTop 10 categories:\n", df['company_category'].value_counts().head(10))

Total companies: 529 | Unique names: 529
Duplicate rows: 0

Missing values:
 company_name         0
company_category     3
description          0
description_short    0
dtype: int64

Top 10 categories:
 company_category
Data/Technology              97
Finance & Investment         75
Business & Legal Services    44
Governance                   43
Healthcare                   40
Geospatial/Mapping           30
Research & Consulting        28
Transportation               28
Energy                       28
Lifestyle & Consumer         25
Name: count, dtype: int64


In [4]:
# Set the item identifier and text feature columns used for recommendations.
item_id_col = 'company_name'
text_cols = ['description', 'company_category', 'description_short']
print(f"Item ID: {item_id_col} | Text features: {text_cols}")
df[[item_id_col] + text_cols].head(3)

Item ID: company_name | Text features: ['description', 'company_category', 'description_short']


,company_name,description,company_category,description_short
0,"3 Round Stones, Inc.",3 Round Stones produces a platform for publish...,Data/Technology,Our Open Source platform is used by the Fortun...
1,48 Factoring Inc.,The company mission is to provide finance to s...,Finance & Investment,48 Factoring Inc. is one of the best financial...
2,5PSolutions,"At 5PSolutions, we wish to make all basic info...",Data/Technology,5PSolutions are artisans of mobile platforms.


- 529 companies, 22 columns, no duplicate rows
- 3 missing values in company_category, none in description
- item name = company_name, text = description + company_category + description_short

### Task 2: Text Preprocessing
fill NaN with "", join the 3 text columns, lowercase, remove punctuation and stopwords -> clean_text

In [5]:
# Fill missing values, join the three text columns, and clean tokens into clean_text.
stop_words = set(stopwords.words('english'))
for col in ['description', 'company_category', 'description_short']:
    df[col] = df[col].fillna('')

df['combined_text'] = df['description'] + ' ' + df['company_category'] + ' ' + df['description_short']

def clean_text(text):
    text = re.sub(r'[^a-z\s]', ' ', text.lower())
    return ' '.join([w for w in text.split() if len(w) > 1 and w not in stop_words])

df['clean_text'] = df['combined_text'].apply(clean_text)

# Verify zero nulls in clean_text and compare original vs cleaned text samples.
print(f"Missing values in clean_text: {df['clean_text'].isna().sum()}")
for i in range(3):
    print(f"\n{df['company_name'].iloc[i]}")
    print(f"Raw  : {df['combined_text'].iloc[i][:100]}...")
    print(f"Clean: {df['clean_text'].iloc[i][:100]}...")

df[['company_name', 'clean_text']].head(5)

Missing values in clean_text: 0

3 Round Stones, Inc.
Raw  : 3 Round Stones produces a platform for publishing data on the Web. 3 Round Stones provides commercia...
Clean: round stones produces platform publishing data web round stones provides commercial support callimac...

48 Factoring Inc.
Raw  : The company mission is to provide finance to small business. We also provide financing to small busi...
Clean: company mission provide finance small business also provide financing small business bad credit basi...

5PSolutions
Raw  : At 5PSolutions, we wish to make all basic information of different categories easily available to vi...
Clean: psolutions wish make basic information different categories easily available via tablets phones data...


,company_name,clean_text
0,"3 Round Stones, Inc.",round stones produces platform publishing data...
1,48 Factoring Inc.,company mission provide finance small business...
2,5PSolutions,psolutions wish make basic information differe...
3,Abt Associates,abt associates mission driven international co...
4,Accela,accela powers thousands services millions tran...


note: I initially used only description but many companies have same words like platform, services, solutions. adding category and short description gave better keywords.

## PART 2 - Text Vectorization
### Task 3: TF-IDF
max_features=5000, ngram_range=(1, 2) so 2 word phrases like "open data" are also kept

In [6]:
# Fit TfidfVectorizer with 5000 max features and (1, 2) n-grams on clean_text.
tfidf = TfidfVectorizer(max_features=5000, ngram_range=(1, 2), stop_words='english')
tfidf_matrix = tfidf.fit_transform(df['clean_text'])
sparsity = (1 - tfidf_matrix.nnz / (tfidf_matrix.shape[0] * tfidf_matrix.shape[1])) * 100

print(f"TF-IDF matrix shape: {tfidf_matrix.shape}")
print(f"Non-zero entries: {tfidf_matrix.nnz} | Sparsity: {sparsity:.2f}%")
features = tfidf.get_feature_names_out()
print(f"First 15 features: {list(features[:15])}")
print(f"Middle 15 features: {list(features[2500:2515])}")

TF-IDF matrix shape: (529, 5000)
Non-zero entries: 25731 | Sparsity: 99.03%
First 15 features: ['ability', 'ability monitor', 'able', 'abt associates', 'abuse', 'academic', 'academic markets', 'academics', 'accela', 'accelerate', 'accelrys', 'accelrys provider', 'accenture', 'access', 'access data']
Middle 15 features: ['open data', 'open government', 'open innovation', 'open marketplace', 'open source', 'opencounter', 'opengov', 'opengov com', 'operate', 'operated', 'operates', 'operating', 'operation', 'operational', 'operations']


- matrix shape (529, 5000), around 99% values are zero (sparse)

### Task 4: Similarity Computation
using cosine similarity because it checks the angle between vectors, not length. so a short and a long description about same topic still get high score.

In [7]:
# Compute pairwise cosine similarity across all company TF-IDF vectors.
similarity_matrix = cosine_similarity(tfidf_matrix, tfidf_matrix)
print(f"Similarity matrix shape: {similarity_matrix.shape}")
print(f"Diagonal check: {similarity_matrix[0, 0]:.4f}, {similarity_matrix[1, 1]:.4f}\n")
for j in range(1, 6):
    print(f"{df['company_name'].iloc[0]} vs {df['company_name'].iloc[j]}: {similarity_matrix[0, j]:.4f}")

Similarity matrix shape: (529, 529)
Diagonal check: 1.0000, 1.0000

3 Round Stones, Inc. vs 48 Factoring Inc.: 0.0000
3 Round Stones, Inc. vs 5PSolutions: 0.0273
3 Round Stones, Inc. vs Abt Associates: 0.0000
3 Round Stones, Inc. vs Accela: 0.0249
3 Round Stones, Inc. vs Accenture: 0.0071


## PART 3 - Recommendation Logic
### Task 5: recommend function
find index of company -> sort its similarity scores -> skip itself -> return top n

In [8]:
# Look up a company index, sort its cosine similarity scores, and return top N peers.
def recommend(item_name, top_n=5):
    q = item_name.strip().lower()
    matches = df[df['company_name'].astype(str).str.lower() == q]
    if matches.empty:
        matches = df[df['company_name'].astype(str).str.lower().str.contains(q, na=False)]
    if matches.empty:
        return pd.DataFrame()
    idx = matches.index[0]
    scores = [s for s in sorted(enumerate(similarity_matrix[idx]), key=lambda x: x[1], reverse=True) if s[0] != idx][:top_n]
    rows = []
    for rank, (i, score) in enumerate(scores, 1):
        desc = str(df['description_short'].iloc[i] or df['description'].iloc[i])[:110]
        rows.append({'Rank': rank, 'Company': df['company_name'].iloc[i], 'Category': df['company_category'].iloc[i], 'Similarity': round(float(score), 4), 'Description': desc})
    return pd.DataFrame(rows)
    
print("Recommendation function ready.")

Recommendation function ready.


In [9]:
# Test recommendations for AccuWeather.
print("Recommendations for AccuWeather:")
print(recommend('AccuWeather', top_n=5).to_string(index=False))

Recommendations for AccuWeather:
 Rank                  Company            Category  Similarity                                                                                                    Description
    1         United Mayflower      Transportation      0.1531              United Mayflower Container Services provides local and long-distance moving and storage services.
    2     State Farm Insurance           Insurance      0.1445                      State Farm is a group of insurance and financial services companies in the United States.
    3 Construction Monitor LLC Housing/Real Estate      0.1444 Construction Monitor provides timely, detailed and accurate building permit information, using state of the ar
    4          Weather Channel                          0.1212 The Weather Channel and weather.com provide a national and local weather forecast for cities, as well as weath
    5             SpotHero.com      Transportation      0.1109 SpotHero provides parking at over 

In [10]:
# Test recommendations for Accenture.
print("Recommendations for Accenture:")
print(recommend('Accenture', top_n=5).to_string(index=False))

Recommendations for Accenture:
 Rank                Company                  Category  Similarity                                                                                                    Description
    1    Booz Allen Hamilton     Research & Consulting      0.1510                      Booz Allen Hamilton provides management consulting, technology, and engineering services.
    2           Oliver Wyman     Research & Consulting      0.1282 Oliver Wyman is a global management consulting firm. We combine industry knowledge with expertise in strategy,
    3          R R Donnelley Business & Legal Services      0.0992 RR Donnelley, a global integrated communications provider, uses digital, conventional, and web-based technolog
    4               Deloitte     Research & Consulting      0.0989 Deloitte works in four areas — audit, financial advisory, tax and consulting — and combines the talents of tho
    5 Energy Solutions Forum                    Energy      0.0967 Energy Solut

In [11]:
# Test recommendations for 48 Factoring Inc.
print("Recommendations for 48 Factoring Inc.:")
print(recommend('48 Factoring Inc.', top_n=5).to_string(index=False))

Recommendations for 48 Factoring Inc.:
 Rank           Company             Category  Similarity                                                                                                    Description
    1        DemystData Finance & Investment      0.2538 DemystData helps financial institutions tap into "Big Data" to make better credit decisions. The company’s pro
    2      Credit Karma Finance & Investment      0.2334 Credit Karma is a web-based credit and financial management service that helps more than 20 million consumers 
    3            OnDeck Finance & Investment      0.2196 OnDeck is a technology-powered Main Street lender that uses big data to deliver financing to small businesses.
    4     Credit Sesame Finance & Investment      0.2046 Credit Sesame is a free tool that provides a free credit score, credit monitoring, and a way to save money on 
    5 J.P. Morgan Chase Finance & Investment      0.1776 J.P. Morgan Chase offers investment banking, financial services 

Observations:
- 48 Factoring Inc. -> DemystData, Credit Karma, OnDeck (all finance)
- Accenture -> Booz Allen Hamilton, Oliver Wyman, Deloitte (all consulting)
- AccuWeather -> United Mayflower is #1 only because both say "united states" and "local", Weather Channel comes at #4
- scores are small (0.1 - 0.25) but ranking is what matters

## PART 4 - Simple App Interface
### Task 6: Streamlit UI
made app.py with company dropdown, slider for number of recommendations and a button. it shows similarity score and the common keywords for each result.

run locally: `streamlit run app.py`

## PART 5 - Git & GitHub
### Task 7
GitHub repo: https://github.com/INEEDTHATGT3/content-based-recommender

In [12]:
# requirements.txt for render
reqs = """streamlit==1.66.1
pandas==3.0.5
numpy==2.5.3
scikit-learn==1.9.1
nltk==3.10.4
"""
with open('requirements.txt', 'w') as f:
    f.write(reqs)
print(reqs)

streamlit==1.66.1
pandas==3.0.5
numpy==2.5.3
scikit-learn==1.9.1
nltk==3.10.4



## PART 6 - Deployment on Render
### Task 8
- new web service on render -> connected github repo -> free instance
- build command: `pip install -r requirements.txt`
- start command: `streamlit run app.py --server.port $PORT --server.address 0.0.0.0`

note: I had to add $PORT and 0.0.0.0, otherwise render can't find the app and deploy fails

### Task 9: Final Validation
Deployed URL: https://content-based-recommender-7aei.onrender.com

- tested AccuWeather, Accenture and 48 Factoring Inc. on the live app, same results as notebook
- learnt: content based filtering doesn't need any user data, only item text. combining 3 text columns worked much better than description alone.